In [ ]:
from loguru import logger
from configs.config import PROJECT_DIR

df = []
try:
    with open(PROJECT_DIR + "/data/original/train.tsv", "r", encoding="utf-8") as f:
        lines = f.readlines()
        for (i, line) in enumerate(lines):
            line = line.strip()
            items = line.split("\t")
            text_a = items[0]
            label = list(map(int, items[1].split(",")))
            df.append({
                "text": text_a, 
                "label": label
            })
        logger.info(f"Load successfully")
except FileNotFoundError as e:
    logger.error(e)


2025-02-25 18:34:09.217 | INFO     | __main__:<module>:17 - Load successfully


In [102]:
corpus = [x["text"] for x in df]

In [105]:
import string
import nltk
nltk.download('stopwords')
nltk.download('wordnet')


from nltk.corpus import stopwords

def remove_stopwords(corpus):
    stop_words = set(stopwords.words('english')) 
    lemmatizer = nltk.WordNetLemmatizer()

    stopped_tokens = []

    for c in corpus:
        lower = c.lower()
        translation_table = str.maketrans('', '', string.punctuation)
        stripped = lower.translate(translation_table)
        tokens = stripped.split(" ")
        filtered_tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]
        stopped_tokens.append(filtered_tokens)
    
    result = [' '.join(token) for token in stopped_tokens]

    return result    

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/haiduong/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/haiduong/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [106]:
res = remove_stopwords(corpus)
res

['favourite food anything didnt cook',
 'everyone think he laugh screwing people instead actually dead',
 'fuck bayless isoing',
 'make feel threatened',
 'dirty southern wanker',
 'omg peyton isnt good enough help u playoff dumbass bronco fan circa december 2015',
 'yes heard abt f bomb thanks reply hubby anxiously wait 😝',
 'need board create bit space name we’ll good',
 'damn youtube outrage drama super lucrative reddit',
 'might linked trust factor friend',
 'demographic don’t know anybody 35 cable tv',
 'aww shell probably come around eventually im sure jealous name mean woman wouldnt lol ',
 'hello everyone im toronto well call visit personal needed',
 'rsleeptrain might time sleep training take look try feel whats right family',
 'name  fucking problem slightly better command english language',
 'shit guess accidentally bought payperview boxing match',
 'thank friend',
 'fucking coward',
 'retardation look like',
 'maybe that’s happened great white houston zoo',
 'never thought 

In [95]:
import numpy as np
import matplotlib.pyplot as plt

# Find 95% length of all corpus
def find_sequence_length(corpus):
    length_corpus = [len(c) for c in corpus]
    percentile_95 = np.percentile(length_corpus, 95)

    return percentile_95

sequence_length = find_sequence_length(res)

In [96]:
print(sequence_length)

87.0


In [98]:
import tensorflow as tf
import keras

corpus = tf.constant(res)

vectorizer = keras.layers.TextVectorization(
    max_tokens=10000,
    output_mode="int",
    output_sequence_length=int(sequence_length),
    standardize=None
)


In [99]:
vectorizer.adapt(res)
output = vectorizer(res)
print(output)

tf.Tensor(
[[1045  403  111 ...    0    0    0]
 [ 106   13   80 ...    0    0    0]
 [  98 9360    1 ...    0    0    0]
 ...
 [ 223  111   39 ...    0    0    0]
 [   3    1 1281 ...    0    0    0]
 [ 237 1579    0 ...    0    0    0]], shape=(43410, 87), dtype=int64)


In [ ]:
from src.dataset.original import OriginalDataset
from configs.config import PROJECT_DIR

org = OriginalDataset(PROJECT_DIR + "/data/original")